In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class Detector:
    def __init__(self, R, r, N):
        self.R = R
        self.r = r
        self.N = N
        self.positions = self._place_pmt()

    def _place_pmt(self):
        points = []
        phi = np.pi * (3. - np.sqrt(5.))  
        for i in range(self.N):
            y = 1 - (i / float(self.N - 1)) * 2  
            radius = np.sqrt(1 - y * y)
            theta = phi * i
            x = np.cos(theta) * radius
            z = np.sin(theta) * radius
            points.append([x, y, z])
        
        points = np.array(points)
        points = points / np.linalg.norm(points, axis=1)[:, np.newaxis] * self.R
        
        for i in range(len(points)):
            for j in range(i + 1, len(points)):
                dist = np.linalg.norm(points[i] - points[j])
                if dist < 2 * self.r:
                    print(f"Внимание: ФЭУ {i} и {j} пересекаются (расстояние {dist:.4f} < {2*self.r})!")
        
        return points


class Source:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_photons(self, E):
        N = int(E * self.Y)
        phi = np.random.uniform(0, 2 * np.pi, N)
        cos_theta = np.random.uniform(-1, 1, N)
        sin_theta = np.sqrt(1 - cos_theta**2)
        x = sin_theta * np.cos(phi)
        y = sin_theta * np.sin(phi)
        z = cos_theta
        return np.column_stack((x, y, z))


def simulate_fast(R=2.0, r=0.1, N_pmt=40, Y=10000, a=0.5, E=1.0,
                  source_pos=np.array([0.0, 0.0, 0.0])):
    det = Detector(R, r, N_pmt)
    source = Source(Y)
    directions = source.generate_photons(E)      
    pmts = det.positions                        

    b = directions @ source_pos                  
    c = source_pos @ source_pos - R**2
    disc = b**2 - c
    valid = disc >= 0
    t = -b[valid] + np.sqrt(disc[valid])
    valid_t = t > 0
    hit_points = source_pos + t[valid_t, None] * directions[valid][valid_t]  # (M, 3)

    if len(hit_points) == 0:
        return 0
        
    diff = hit_points[:, None, :] - pmts[None, :, :]
    dists = np.linalg.norm(diff, axis=2)

    mask = dists <= r
    electrons = np.random.poisson(a, size=mask.shape) * mask
    return int(electrons.sum())

R = 2.0
r = 0.1
N_pmt = 40
Y = 10000
a = 0.5
E = 1.0

Q_sim = simulate_fast(R, r, N_pmt, Y, a, E)
Q_expected = a * (E * Y) * (N_pmt * r**2 / R**2)

print(f"Сгенерировано фотонов: {int(E * Y)}")
print(f"Полный собранный заряд Q (симуляция): {Q_sim}")
print(f"Ожидаемый заряд Q (теория): {Q_expected:.1f}")
print(f"Отклонение: {abs(Q_sim - Q_expected) / Q_expected * 100:.1f}%")


print("\n1. Моделирование 1000 событий для E = 1.0 МэВ...")
n_events = 1000
charges = np.array([simulate_fast(R, r, N_pmt, Y, a, E) for _ in range(n_events)])

plt.figure(figsize=(8, 4))
plt.hist(charges, bins=30, alpha=0.7, edgecolor='black')
plt.xlabel("Заряд Q (число фотоэлектронов)")
plt.ylabel("Число событий")
plt.title("Распределение заряда для E = 1.0 МэВ")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Средний заряд: {charges.mean():.1f} ± {charges.std():.1f}")
print(f"Относительная ширина: {charges.std() / charges.mean():.3f}")

print("\n2. Исследование зависимости ширины от энергии...")
energies = np.array([0.2, 0.5, 1.0, 2.0])
widths = []
for E_val in energies:
    ch = np.array([simulate_fast(R, r, N_pmt, Y, a, E_val) for _ in range(200)])
    w = ch.std() / ch.mean()
    widths.append(w)
    print(f"E = {E_val:.1f} МэВ: среднее = {ch.mean():.1f}, ширина = {w:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(energies, widths, 'o-')
plt.xlabel("Энергия E, МэВ")
plt.ylabel("Относительная ширина σ/μ")
plt.title("Зависимость относительной ширины от энергии")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Сгенерировано фотонов: 10000
Полный собранный заряд Q (симуляция): 111
Ожидаемый заряд Q (теория): 500.0
Отклонение: 77.8%

1. Моделирование 1000 событий для E = 1.0 МэВ...
